# Analysing Knowledge Graphs

Creating a knowledge graph is a first step, but we will want to use it to ask questions about the problem it is modelling. Two common questions that we will want to answer are:

* For a given entity $E$, what other things are "nearby" in the graph and therefore related to $E$.

* Can we identify potential relations between entity $E$ and some other entity $F$ in the graph, where a direction relationship between them does not already exist?

Central to answering these questions will be the search techniques we developed in the context of logic. Just as we can search on the graph induced my mapping a logical knowledge base onto a rule-based system, so we can apply those techniques to knowledge graphs.

In [1]:
from owlready2 import *
onto = get_ontology("./teaching.rdf").load()

print(onto["Staff"])

teaching.Staff


## Basic searches in Owlready2

OWLReady2 incorporate some basic search functionality, described in the documentation at https://owlready2.readthedocs.io/en/latest/onto.html. Let us explore the capabilities of this. First, a very simple search to find the node corresponding to a person with a specific name

In [17]:
onto.search(person_name = "Prof Iain Styles")

[teaching.sta05]

We can also use wildcards if we are not sure what the exact value is (did we use Prof/Prof./Professor?):

In [19]:
x = onto.search(person_name = "*Styles")
print(x[0].person_name)

['Prof Iain Styles']


Note that this returns a list and so supports the return of multiple entries, for example, we can get all professorial staff:

In [23]:
profs = onto.search(person_name="Prof*")
for i in profs:
    print(i.person_name)

['Prof Hui Wang']
['Prof Iain Styles']
["Prof Maire O'Neill"]


Notice that this search, by attribute, is over all entities with attribute `person_name`, so a very generic search will return both `Staff` and `Student`. For example, all name containing the letter "y"

In [26]:
x = onto.search(person_name="*y*")
for i in x:
    print(i.person_name)

['Dr Barry Devereux']
['Prof Iain Styles']
['Dr Ciara Rafferty']
['Dr Ayesha Khalid']
['Dr Amy Liu']
['John McCarthy']
['Raj Reddy']
['Geoffrey Hinton']
['Annie Easley']


We can limit this by nesting searches, in this case adding a search for the `type` of the entity:

In [34]:
x = onto.search(is_a = onto.Staff, person_name="*y*")
for i in x:
    print(i.person_name)

['Dr Barry Devereux']
['Prof Iain Styles']
['Dr Ciara Rafferty']
['Dr Ayesha Khalid']
['Dr Amy Liu']


This has shown us how to query the graph by the attributes of its entities. What about querying it by relationship? This is trivial if the relationship come "from" the entity we want to query, so we easily find, for example, the program a specific student is enrolled on , because the relationship is `is_enrolled_on`:

In [122]:
x = onto.search_one(person_name='*Hinton')
print(x.is_enrolled_on[0].program_title)

['MSc Artificial Intelligence']



What if we wanted to turn this around though? Let's say we wanted get all the students enrolled on a program? We will do this with a nested query in which we first search for the programme we want, and then search for the students that `is_enrolled_on` that program:

In [124]:
x = onto.search(is_enrolled_on = onto.search(program_title="*Cyber*"))
print(x)

[teaching.stu06, teaching.stu07, teaching.stu08, teaching.stu09, teaching.stu10]


This is about the limit of how complex we can make these queries. To do more sophisticated things we need to use SPARQL. Let's recreate some of these queris in SPARQL and then do some more complex ones. Let's start with search for a person by name:

In [94]:
x = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?s WHERE
    {
        ?s ONTO:person_name "Prof Iain Styles"
    }
    """))

print(x[0][0].person_name[0])

Prof Iain Styles


What about with a wildcard? This is not so easy and we have to do this using a `FILTER`

In [95]:
x = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?s ?n WHERE
    {
        ?s ONTO:person_name ?n
        FILTER(CONTAINS(?n, "Styles")).
    }
    """))

print(x)

[[teaching.sta05, 'Prof Iain Styles']]


We can similarly use a filter to get all Professorial staff by filtering the start of the string:

In [96]:
x = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?s ?n WHERE
    {
        ?s ONTO:person_name ?n
        FILTER(STRSTARTS(?n, "Prof")).
    }
    """))

print(x)

[[teaching.sta04, 'Prof Hui Wang'], [teaching.sta05, 'Prof Iain Styles'], [teaching.sta12, "Prof Maire O'Neill"]]


And all persons containing the letter 'y':

In [97]:
x = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?s ?n WHERE
    {
        ?s ONTO:person_name ?n
        FILTER(CONTAINS(?n, "y")).
    }
    """))

print(x)

[[teaching.sta01, 'Dr Barry Devereux'], [teaching.sta05, 'Prof Iain Styles'], [teaching.sta09, 'Dr Ciara Rafferty'], [teaching.sta10, 'Dr Ayesha Khalid'], [teaching.sta11, 'Dr Amy Liu'], [teaching.stu02, 'John McCarthy'], [teaching.stu03, 'Raj Reddy'], [teaching.stu04, 'Geoffrey Hinton'], [teaching.stu06, 'Annie Easley']]


Limit this to `Staff` only:

In [98]:
x = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?s ?n WHERE
    {
        ?s rdf:type ONTO:Staff
        ?s ONTO:person_name ?n
        FILTER(CONTAINS(?n, "y")).
    }
    """))

print(x)

[[teaching.sta01, 'Dr Barry Devereux'], [teaching.sta05, 'Prof Iain Styles'], [teaching.sta09, 'Dr Ciara Rafferty'], [teaching.sta10, 'Dr Ayesha Khalid'], [teaching.sta11, 'Dr Amy Liu']]


Now let's find who teaches a specific module

In [ ]:
x = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?s ?m WHERE
    {
        ?s rdf:type ONTO:Staff
        ?m rdf:type ONTO:Module
        ?m ONTO:is_taught_by ?s
        ?m ONTO:module_title "Knowledge Engineering"
    }
    """))

print(x)

[[teaching.sta05, teaching.mod03]]


We could also do this using a filter. This has the advantage that the filter can be set to match the query term exactly, whereas the method we have used in the cell above will match "Knowledge Engineering*"

In [107]:
x = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?s ?m WHERE
    {
        ?s rdf:type ONTO:Staff
        ?m rdf:type ONTO:Module
        ?m ONTO:is_taught_by ?s
        ?m ONTO:module_title $t
        FILTER($t="Knowledge Engineering")
    }
    """))

print(x)

[[teaching.sta05, teaching.mod03]]


If we know exactly what we are looking for, we can construct the query relatively easily. What if we want to construct a more general query, for example, to find all relations of *any* type that are outgoing from a particular entity? Can we do this using some form of wildcard? Yes, we can, but it is not completely straightforward. We can easily find all the relations but we need to filter them based on their prefix in the ontology:

In [151]:
x = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?m $r $s WHERE
    {
        ?m rdf:type ONTO:Module
        ?m ONTO:module_title ?t
        $m $r $s
        FILTER($t="Machine Learning")
        FILTER(STRSTARTS(STR(?r), "http://www.dummy.info/new.owl#"))
        FILTER(STRSTARTS(STR(?s), "http://www.dummy.info/new.owl#"))

    }
    """))

print(x)

[[teaching.mod02, teaching.is_taught_by, teaching.sta04]]


If we don't include this additional filter we will also get the object attributes rather than just the relations.

We can fairly easily turn this around to find all incoming relations:

In [149]:
x = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?s ?r $m WHERE
    {
        ?m rdf:type ONTO:Module
        ?m ONTO:module_title ?t
        $s $r $m
        FILTER($t="Machine Learning")
        FILTER(STRSTARTS(STR(?r), "http://www.dummy.info/new.owl#"))
        FILTER(STRSTARTS(STR(?s), "http://www.dummy.info/new.owl#"))

    }
    """))

print(x)

[[teaching.pro01, teaching.has_module, teaching.mod02], [teaching.pro02, teaching.has_module, teaching.mod02], [teaching.pro03, teaching.has_module, teaching.mod02]]


We can then combine these two queries to construct the graph neighbourhood. Note that we wouldn't want to use the built-in `UNION` function in SPARQL to do this because we would lose the directionality of the edges.

SPARQL is a rich, expressive, and flexible language 

Disease "Prostatic Neoplasms"
Extract its 1- and 2- neighbourhoods.

Find shortest pathway to disease "Prostatis" (there is a pathway via Gene CXCL12 and Gene NPEPPS). Might be others, check for direct connection.

Question - would you use BFS or DFS here? What are the trade-offs?

In [ ]:
# Placeholder for code.

## Even more advanced aspects of OWLReady2

OWLReady2 incorporates a wide range of extra functionality that we will not cover during this module. This includes:

* Subclasses: enables specialisation of classes. For example, `Student` could have subclasses `PartTimeStudent` and `FullTimeStudent` that would inherit from `Student`
* Restrictions: limits the possible properties that an entity can have. For example, a `FullTimeStudent` must be enrolled for 1 year wheras a `PartTimeStudent` must be enrolled for 2 years.
* Reasoning: re-assigns individuals to classes based on their properties. An entity that is of type `Student` and is enrolled for 2 years would be re-classified as a `PartTimeStudent`.
* Open vs closed-world reasoning: in an open world, anything can happen that is not explicitly prevented, whereas in a closed world, only those things that are explicitly allowed can happen.

These are described in detail and with examples in the documentation at https://owlready2.readthedocs.io/en/latest/intro.html.
